# VisionStream AI — Sprint 3: Remote NVIDIA GPU Lab
## 04. TensorRT Feasibility, Compatibility Check & Engine Build Probe

**Tujuan:** Menguji kelayakan instalasi TensorRT, kompatibilitas arsitektur GPU yang teralokasi, serta melakukan uji coba konversi model YOLOv8n ke engine TensorRT (`.engine`).

> **Catatan Teknis Penting:** TensorRT 10.x/11.x memerlukan arsitektur GPU dengan Compute Capability minimal 7.0 (Volta/Turing) ke atas. GPU tipe Tesla P100 (Pascal SM 6.0) **TIDAK DIDUKUNG** oleh TensorRT 10+/11+ modern.

### Step 1: Evaluate Allocated GPU Compute Capability & TensorRT Eligibility

In [ ]:
import torch

assert torch.cuda.is_available(), "NVIDIA GPU is required!"
device_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
compute_cap = float(f"{major}.{minor}")

print(f"Allocated GPU: {device_name}")
print(f"Compute Capability: {compute_cap} (SM {major}{minor})")

# Verification criteria
if compute_cap < 7.0:
    print(f"WARNING: GPU Compute Capability {compute_cap} is BELOW 7.0!")
    print("Modern TensorRT (10.x / 11.x) drops support for SM < 7.0 (e.g. Pascal P100).")
    print("TensorRT Engine compilation will likely FAIL or require deprecated TensorRT 8.x.")
else:
    print("ELIGIBILITY CHECK: PASS! GPU supports modern TensorRT (Turing/Ampere/Ada Lovelace).")

### Step 2: Install TensorRT Dependencies in Notebook Environment

In [ ]:
# Attempt installing tensorrt via pip
!pip install --quiet tensorrt ultralytics

### Step 3: Probe TensorRT Engine Conversion (YOLOv8 Engine Export)
*Cell ini menguji ekspor langsung model kanonikal YOLOv8n ke engine TensorRT FP16 via Ultralytics.*

In [ ]:
import os, json, time
import numpy as np
from ultralytics import YOLO

MODEL_PT = "yolov8n.pt"
assert os.path.exists(MODEL_PT), f"{MODEL_PT} must exist."

trt_export_success = False
trt_results = {}

try:
    model = YOLO(MODEL_PT)
    print("Attempting TensorRT engine export (format='engine', half=True, imgsz=640)...")
    t0 = time.perf_counter()
    engine_path = model.export(format="engine", half=True, imgsz=640, device=0)
    t1 = time.perf_counter()
    print(f"TensorRT Engine exported successfully in {t1 - t0:.2f}s: {engine_path}")
    trt_export_success = True
    
    # Benchmark TensorRT Engine
    trt_model = YOLO(engine_path)
    dummy_frame = np.random.randint(0, 256, (640, 640, 3), dtype=np.uint8)
    
    print("Running 20 warmup iterations...")
    for _ in range(20):
        _ = trt_model.predict(dummy_frame, device="cuda:0", verbose=False)
    
    print("Running 200 measured benchmark iterations...")
    latencies = []
    for _ in range(200):
        tb0 = time.perf_counter()
        _ = trt_model.predict(dummy_frame, device="cuda:0", verbose=False)
        torch.cuda.synchronize()
        tb1 = time.perf_counter()
        latencies.append((tb1 - tb0) * 1000.0)
        
    mean_lat = float(np.mean(latencies))
    fps = float(1000.0 / mean_lat)
    print(f"TensorRT FP16 Mean Latency: {mean_lat:.2f} ms ({fps:.2f} FPS)")
    
    trt_results = {
        "status": "SUCCESS",
        "engine_path": str(engine_path),
        "mean_latency_ms": mean_lat,
        "fps": fps
    }
except Exception as e:
    print("TensorRT export/inference failed with error:", e)
    trt_results = {
        "status": "FAILED",
        "error": str(e)
    }

with open("tensorrt_feasibility_results.json", "w") as f:
    json.dump(trt_results, f, indent=2)
print("TensorRT feasibility results saved to tensorrt_feasibility_results.json")